{{ badge }}

In [1]:
#MODEL CLASS = creating a class for the model which inherits from nn.Module
import torch
import torch.nn as nn
#use if nn.doule for all functioning
class Model(nn.Module):
    #num_fetures is the number of features in the dataset and output is 1 because we are doing regression
    def __init__(self, num_features):
        super(Model, self).__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, features):
        output = self.linear(features)
        output = self.sigmoid(output)
        return output

In [2]:
#create dataset 
features = torch.rand(10,5) #10 samples and 5 features
#create object model
model = Model(features.shape[1])
#call forward pass
model_output = model(features)
print(model_output)


tensor([[0.4344],
        [0.4892],
        [0.3723],
        [0.4502],
        [0.5479],
        [0.4998],
        [0.3874],
        [0.4609],
        [0.5247],
        [0.4417]], grad_fn=<SigmoidBackward0>)


In [ ]:
#model weights and bias
print("model weights: ", model.linear.weight)
print("model bias: ", model.linear.bias)


model weights:  Parameter containing:
tensor([[ 0.1972, -0.0635, -0.3972,  0.4321,  0.4257]], requires_grad=True)
model bias:  Parameter containing:
tensor([-0.4234], requires_grad=True)


In [4]:
#vizualize model
!pip install torchinfo
from torchinfo import summary
summary(model, input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 1]                   6
├─Sigmoid: 1-2                           [10, 1]                   --
Total params: 6
Trainable params: 6
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [5]:
#second nn with 3 relu neuron and one sigmoid output
class Model2(nn.Module):
    def __init__(self, num_features):
        super(Model2, self).__init__()
        self.linear1 = nn.Linear(num_features, 3)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(3, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, features):
        output = self.linear1(features)
        output = self.relu(output)
        output = self.linear2(output)
        output = self.sigmoid(output)
        return output

In [9]:
#create dataset 
features = torch.rand(10,5) #10 samples and 5 features
#create object model
model = Model2(features.shape[1])
#call forward pass
model_output = model(features)
print(model_output)

tensor([[0.5997],
        [0.6028],
        [0.6279],
        [0.6120],
        [0.6031],
        [0.6213],
        [0.6085],
        [0.5866],
        [0.6051],
        [0.6363]], grad_fn=<SigmoidBackward0>)


In [7]:
#much more easy way using sequential container 
class Model2(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 3),
            nn.ReLU(),
            nn.Linear(3, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        output = self.network(features)
        return output

In [10]:
#create dataset 
features = torch.rand(10,5) #10 samples and 5 features
#create object model
model = Model2(features.shape[1])
#call forward pass
model_output = model(features)
print(model_output)

tensor([[0.4181],
        [0.4269],
        [0.4095],
        [0.4048],
        [0.4157],
        [0.4008],
        [0.4260],
        [0.4198],
        [0.4082],
        [0.4142]], grad_fn=<SigmoidBackward0>)


In [ ]:
#autobuilt loss function
loss_fn = nn.BCELoss()
type(loss_fn)

torch.nn.modules.loss.BCELoss

In [12]:
#optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
optimizer.zero_grad()
optimizer.step()

In [1]:
#dataset and dataloader = a way to load data in batches and shuffle it
#dataset = how data is structured and returned
#dataloader = wraps dataset and handles batching, shuffling and loading data in parallel using multiprocessing workers

from sklearn.datasets import make_classification
import torch
X, y = make_classification(
    n_samples=10,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=2, 
    random_state=42
    )
print(X)
print(y)
print(X.shape, y.shape)

[[ 1.06833894 -0.97007347]
 [-1.14021544 -0.83879234]
 [-2.8953973   1.97686236]
 [-0.72063436 -0.96059253]
 [-1.96287438 -0.99225135]
 [-0.9382051  -0.54304815]
 [ 1.72725924 -1.18582677]
 [ 1.77736657  1.51157598]
 [ 1.89969252  0.83444483]
 [-0.58723065 -1.97171753]]
[1 0 0 0 0 1 1 1 1 0]
(10, 2) (10,)


In [2]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)
print(X)
print(y)

tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388],
        [-2.8954,  1.9769],
        [-0.7206, -0.9606],
        [-1.9629, -0.9923],
        [-0.9382, -0.5430],
        [ 1.7273, -1.1858],
        [ 1.7774,  1.5116],
        [ 1.8997,  0.8344],
        [-0.5872, -1.9717]])
tensor([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])


In [ ]:
#main thing
from torch.utils.data import Dataset, DataLoader

class customDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return self.features.shape[0]
#note- transformation of data can be done before __getitem__ method
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

dataset = customDataset(X, y)
print(len(dataset))
dataset[0]

10


(tensor([ 1.0683, -0.9701]), tensor(1))

In [8]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)
for batch_features, batch_labels in dataloader:
    print(batch_features)
    print(batch_labels)
    print("-" * 50)

tensor([[-1.9629, -0.9923],
        [-0.5872, -1.9717]])
tensor([0, 0])
--------------------------------------------------
tensor([[ 1.7774,  1.5116],
        [-1.1402, -0.8388]])
tensor([1, 0])
--------------------------------------------------
tensor([[-2.8954,  1.9769],
        [ 1.7273, -1.1858]])
tensor([0, 1])
--------------------------------------------------
tensor([[-0.9382, -0.5430],
        [ 1.0683, -0.9701]])
tensor([1, 1])
--------------------------------------------------
tensor([[-0.7206, -0.9606],
        [ 1.8997,  0.8344]])
tensor([0, 1])
--------------------------------------------------


In [9]:
#all this process is sequential and time taking so we can use num_workers to load data in parallel using multiprocessing workers
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=2)
for batch_features, batch_labels in dataloader:
    print(batch_features)
    print(batch_labels)
    print("-" * 50)

tensor([[ 1.0683, -0.9701],
        [ 1.8997,  0.8344]])
tensor([1, 1])
--------------------------------------------------
tensor([[ 1.7273, -1.1858],
        [-0.7206, -0.9606]])
tensor([1, 0])
--------------------------------------------------
tensor([[-1.9629, -0.9923],
        [ 1.7774,  1.5116]])
tensor([0, 1])
--------------------------------------------------
tensor([[-0.9382, -0.5430],
        [-2.8954,  1.9769]])
tensor([1, 0])
--------------------------------------------------
tensor([[-0.5872, -1.9717],
        [-1.1402, -0.8388]])
tensor([0, 0])
--------------------------------------------------


In [10]:
#sampler in dataloader is used to specify the strategy to draw samples from the dataset. It can be used to create custom sampling strategies, such as weighted sampling or stratified sampling. By default, dataloader uses a random
#example of weighted random sampler
from torch.utils.data import WeightedRandomSampler
weights = [0.1, 0.2, 0.3, 0.4]  # Example weights for each sample
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
dataloader = DataLoader(dataset, batch_size=2, sampler=sampler)
for batch_features, batch_labels in dataloader:
    print(batch_features)
    print(batch_labels)
    print("-" * 50)


tensor([[-2.8954,  1.9769],
        [-0.7206, -0.9606]])
tensor([0, 0])
--------------------------------------------------
tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388]])
tensor([1, 0])
--------------------------------------------------


In [12]:
#collate_fn in dataloader is functon used to specify how to combine datset into a single batch. we can customize the way data is collated into batches by defining a custom collate function and passing it to the dataloader. This is useful when dealing with variable-length sequences or when you want to apply specific transformations to the data before batching.
#example = textual data where each sample can have different length and we want to pad them (padding) to the same length before batching
#simple example of custom input data 
data = [
    {"text": "Hello world", "label": 0},
    {"text": "PyTorch is great", "label": 1},
    {"text": "I love machine learning", "label": 1}
]
from torch.utils.data import DataLoader
def custom_collate_fn(batch):
    texts = [item["text"] for item in batch]
    labels = [item["label"] for item in batch]
    # Here you can add padding or other transformations to the texts if needed
    return texts, torch.tensor(labels)


dataloader = DataLoader(data, batch_size=2, collate_fn=custom_collate_fn)
for batch_texts, batch_labels in dataloader:
    print(batch_texts)
    print(batch_labels)
    print("-" * 50)


['Hello world', 'PyTorch is great']
tensor([0, 1])
--------------------------------------------------
['I love machine learning']
tensor([1])
--------------------------------------------------


In [13]:
#pin_memory = true, improve gpu speed when working on gpu
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)

#drop_last = true, drop the last batch if it is not full
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)